# 🏛️ Web Scraping SSI MEF — Extracción de Valores

**Fuente:** https://ofi5.mef.gob.pe/ssi/

Este notebook extrae los siguientes datos por CUI de Invierte.pe:
- **Parte 1:** Costo Total, PIA, PIM, Devengado Acumulado, Devengado 2026 y Fechas
- **Parte 2:** % Avance Físico

---
### 📋 Instrucciones de uso
1. Coloca tu archivo `CUI.xlsx` en la **misma carpeta** que este notebook
2. El archivo debe tener una columna llamada **`CUI`**
3. Edita **solo** la celda de Configuración (PASO 2)
4. Ejecuta las celdas en orden con **Shift+Enter**
5. El progreso se guarda automáticamente; si se interrumpe, re-ejecuta para continuar

---
## 📦 PARTE 1 — Costo, PIM, PIA, Devengado y Fechas
### Extrae datos del modal *Ver Resumen* de https://ofi5.mef.gob.pe/ssi/

### ⚙️ PASO 1 — Instalación de dependencias
> Ejecuta esta celda solo la primera vez o en un entorno nuevo.

In [ ]:
!pip install selenium webdriver-manager openpyxl pandas

### ⚙️ PASO 2 — Configuración
> **⚠️ Esta es la ÚNICA celda que debes editar.**
> Cambia el nombre del archivo de entrada y ajusta los parámetros si lo necesitas.

In [ ]:
from pathlib import Path

# ================================================================
# CONFIGURACIÓN — Edita solo esta sección
# ================================================================

# Carpeta base: detecta automáticamente la carpeta del notebook
CARPETA_BASE = Path.cwd()

# --- Archivos ---
# Coloca tu Excel en la misma carpeta que este notebook
NOMBRE_ARCHIVO_ENTRADA = "CUI.xlsx"             # <- Cambia si tu archivo tiene otro nombre
NOMBRE_ARCHIVO_SALIDA  = "resultados_SSI.xlsx"  # <- Nombre del archivo de resultados

RUTA_ENTRADA = CARPETA_BASE / NOMBRE_ARCHIVO_ENTRADA
RUTA_SALIDA  = CARPETA_BASE / NOMBRE_ARCHIVO_SALIDA

# --- Parámetros del scraping ---
MAX_REINTENTOS = 2     # Reintentos por CUI fallido
MODO_VISIBLE   = True  # True = Chrome visible | False = modo silencioso (headless)
GUARDAR_CADA   = 5     # Guardar progreso cada N CUIs

# --- Timeouts en segundos ---
TIMEOUT_PAGINA   = 20
TIMEOUT_ELEMENTO = 10
TIMEOUT_MODAL    = 10

# ================================================================
# Verificación automática
# ================================================================
print(f"📁 Carpeta de trabajo : {CARPETA_BASE}")
print(f"📥 Archivo de entrada : {NOMBRE_ARCHIVO_ENTRADA}")
print(f"📤 Archivo de salida  : {NOMBRE_ARCHIVO_SALIDA}")
print()
if RUTA_ENTRADA.exists():
    print(f"✅ '{NOMBRE_ARCHIVO_ENTRADA}' encontrado — listo para continuar")
else:
    print(f"❌ ERROR: No se encontró '{NOMBRE_ARCHIVO_ENTRADA}' en:")
    print(f"   {CARPETA_BASE}")
    print("   Coloca el archivo en esa carpeta y vuelve a ejecutar esta celda.")

### ⚙️ PASO 3 — Importar librerías

In [ ]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as Wait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager

print("✅ Librerías importadas correctamente")

### ⚙️ PASO 4 — Funciones de checkpoint (guardar y reanudar progreso)

In [ ]:
def cargar_progreso():
    """Carga el progreso previo si existe el archivo de salida."""
    if RUTA_SALIDA.exists():
        try:
            df = pd.read_excel(RUTA_SALIDA)
            print(f"📥 Progreso encontrado: {len(df)} CUIs ya procesados")
            return df.to_dict('records')
        except Exception as e:
            print(f"⚠️ No se pudo leer el progreso anterior: {e}")
    return []

def obtener_pendientes(completa, procesados):
    """Calcula CUIs pendientes o con datos NO DISPONIBLE para reintentar."""
    if not procesados:
        print(f"📋 Todos los CUIs pendientes: {len(completa)}")
        return completa, {}

    procesados_dict = {str(r['CUI']): r for r in procesados}

    cuis_nuevos = [cui for cui in completa if str(cui) not in procesados_dict]

    cuis_reintentar = [
        cui for cui in completa
        if str(cui) in procesados_dict
        and str(procesados_dict[str(cui)].get('Costo Inversión Total (a)', '')).strip().upper()
            in ('NO DISPONIBLE', '')
    ]

    pendientes = cuis_nuevos + cuis_reintentar
    if pendientes:
        print(f"📋 Nuevos: {len(cuis_nuevos)} | Reintentos: {len(cuis_reintentar)} | Total pendiente: {len(pendientes)}")
    return pendientes, procesados_dict

def guardar(resultados):
    """Guarda el progreso en el archivo de salida."""
    try:
        pd.DataFrame(resultados).to_excel(RUTA_SALIDA, index=False)
    except Exception as e:
        print(f"⚠️ Error al guardar: {e}")

print("✅ Funciones de checkpoint listas")

### ⚙️ PASO 5 — Configurar el navegador Chrome

In [ ]:
def crear_driver():
    """Crea Chrome con configuración optimizada para scraping."""
    service = Service(ChromeDriverManager().install())
    options = webdriver.ChromeOptions()

    if not MODO_VISIBLE:
        options.add_argument("--headless=new")
    else:
        options.add_argument("--start-maximized")

    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-extensions")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-logging")
    options.add_argument("--log-level=3")
    options.add_argument("--silent")

    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.default_content_setting_values.media_stream": 2,
        "profile.default_content_setting_values.geolocation": 2,
    }
    options.add_experimental_option("prefs", prefs)
    options.add_experimental_option('excludeSwitches', ['enable-logging'])
    options.page_load_strategy = 'normal'

    driver = Chrome(service=service, options=options)
    driver.set_page_load_timeout(TIMEOUT_PAGINA)
    driver.set_script_timeout(15)
    return driver

print("✅ Función crear_driver() lista")

### ⚙️ PASO 6 — Funciones de scraping del modal SSI

In [ ]:
def cerrar_ventanas_emergentes(driver):
    """Cierra modales Bootstrap que puedan estar abiertos."""
    try:
        driver.execute_script("""
            document.querySelectorAll('.modal.show').forEach(m => {
                m.style.display = 'none';
                m.classList.remove('show');
            });
            document.querySelectorAll('.modal-backdrop').forEach(b => b.remove());
            document.body.classList.remove('modal-open');
            document.body.style.overflow = '';
            document.body.style.paddingRight = '';
        """)
    except:
        pass

def esperar_modal_visible(driver, timeout=10):
    """Espera a que el modal 'Ver Resumen' esté visible y con contenido."""
    try:
        Wait(driver, timeout).until(
            EC.visibility_of_element_located((By.ID, "divResumenCont"))
        )
        time.sleep(0.3)
        tiene_contenido = driver.execute_script("""
            var modal = document.getElementById('divResumenCont');
            if (!modal || modal.children.length === 0 || modal.offsetParent === null) return false;
            return document.getElementById('td_mtototal2_r') !== null &&
                   document.getElementById('val_pim_r') !== null;
        """)
        if not tiene_contenido:
            time.sleep(0.5)
            tiene_contenido = driver.execute_script(
                "return document.getElementById('td_mtototal2_r') !== null;"
            )
        return tiene_contenido
    except TimeoutException:
        return False
    except Exception as e:
        print(f"⚠️ Error: {str(e)[:30]}", end=" ")
        return False

def extraer_datos_modal(driver):
    """Extrae todos los valores del modal 'Ver Resumen' del SSI."""
    try:
        datos = driver.execute_script("""
            function getText(id) {
                try {
                    var e = document.getElementById(id);
                    if (!e) return 'NO DISPONIBLE';
                    return (e.textContent || e.innerText || '').trim() || 'NO DISPONIBLE';
                } catch(err) { return 'NO DISPONIBLE'; }
            }
            function getPIA2026() {
                try {
                    var tabla = document.getElementById('tb_hist_anual_res');
                    if (!tabla) return 'NO DISPONIBLE';
                    var filas = tabla.querySelectorAll('tr');
                    for (var i = 0; i < filas.length; i++) {
                        var celdas = filas[i].querySelectorAll('td');
                        if (celdas.length >= 2 && celdas[0].textContent.trim() === '2026')
                            return celdas[1].textContent.trim() || 'NO DISPONIBLE';
                    }
                } catch(err) {}
                return 'NO DISPONIBLE';
            }
            return {
                costo_total:     getText('td_mtototal2_r'),
                pia_2026:        getPIA2026(),
                pim_2026:        getText('val_pim_r'),
                devengado_acum:  getText('val_efin_r'),
                devengado_2026:  getText('val_avan_r'),
                fecha_inicio:    getText('fec_iniejec_r'),
                fecha_fin:       getText('fec_finejec_r'),
                fecha_primer_dev: getText('pridev_r')
            };
        """)
        valores_validos = sum(1 for v in datos.values() if v != 'NO DISPONIBLE')
        if valores_validos < 3:
            print(f"⚠️ Pocos datos ({valores_validos}/8)", end=" ")
            return None
        return datos
    except Exception as e:
        print(f"⚠️ Error extrayendo datos: {str(e)[:30]}", end=" ")
        return None

print("✅ Funciones de scraping del modal listas")

### ⚙️ PASO 7 — Función de procesamiento por CUI y helpers de resultados

In [ ]:
def procesar_cui(driver, cui, intento=1):
    """Busca el CUI en el SSI, abre el modal 'Ver Resumen' y extrae los datos."""
    try:
        driver.get("https://ofi5.mef.gob.pe/ssi/")

        input_box = Wait(driver, TIMEOUT_ELEMENTO).until(
            EC.element_to_be_clickable((By.ID, "txt_cu"))
        )
        input_box.clear()
        input_box.send_keys(cui)

        btn_buscar = Wait(driver, TIMEOUT_ELEMENTO).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "btn_bus"))
        )
        btn_buscar.click()

        Wait(driver, TIMEOUT_ELEMENTO).until(
            EC.presence_of_element_located((By.ID, "td_cu"))
        )
        time.sleep(0.3)
        cerrar_ventanas_emergentes(driver)
        time.sleep(0.2)

        try:
            btn_resumen = Wait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "img[src*='resumen.png']"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", btn_resumen)
            time.sleep(0.2)
            driver.execute_script("arguments[0].click();", btn_resumen)
        except:
            driver.execute_script("""
                var imgs = document.querySelectorAll('img[src*="resumen.png"]');
                if (imgs.length > 0) imgs[0].click();
            """)

        time.sleep(0.5)

        if not esperar_modal_visible(driver, TIMEOUT_MODAL):
            raise Exception("Modal no cargó a tiempo")

        datos = extraer_datos_modal(driver)
        if not datos:
            raise Exception("Sin datos válidos en el modal")

        try:
            driver.execute_script("""
                var btn = document.querySelector('button[data-bs-dismiss="modal"]');
                if (btn) btn.click();
            """)
            time.sleep(0.2)
        except:
            pass

        return datos

    except Exception as e:
        if intento < MAX_REINTENTOS:
            print(f"🔄{intento+1}...", end=" ")
            time.sleep(1)
            cerrar_ventanas_emergentes(driver)
            return procesar_cui(driver, cui, intento + 1)
        else:
            raise e

def actualizar_resultado(resultados, cui, datos):
    """Actualiza un CUI existente en la lista de resultados o agrega uno nuevo."""
    registro = {
        "CUI": cui,
        "Costo Inversión Total (a)":      datos['costo_total'],
        "PIA 2026":                        datos['pia_2026'],
        "PIM 2026 (c)":                    datos['pim_2026'],
        "Devengado Acumulado al 2026 (b)": datos['devengado_acum'],
        "Devengado 2026 (d)":              datos['devengado_2026'],
        "Fecha Inicio Ejecución":          datos['fecha_inicio'],
        "Fecha Fin Ejecución":             datos['fecha_fin'],
        "Fecha del Primer Devengado":      datos['fecha_primer_dev'],
    }
    for i, r in enumerate(resultados):
        if str(r['CUI']) == str(cui):
            resultados[i] = registro
            return
    resultados.append(registro)

def agregar_fallo(resultados, cui):
    """Registra un CUI como fallido si no existe aún en resultados."""
    if any(str(r['CUI']) == str(cui) for r in resultados):
        return
    resultados.append({
        "CUI": cui,
        "Costo Inversión Total (a)":      "NO DISPONIBLE",
        "PIA 2026":                        "NO DISPONIBLE",
        "PIM 2026 (c)":                    "NO DISPONIBLE",
        "Devengado Acumulado al 2026 (b)": "NO DISPONIBLE",
        "Devengado 2026 (d)":              "NO DISPONIBLE",
        "Fecha Inicio Ejecución":          "NO DISPONIBLE",
        "Fecha Fin Ejecución":             "NO DISPONIBLE",
        "Fecha del Primer Devengado":      "NO DISPONIBLE",
    })

print("✅ Funciones de procesamiento listas")

### ▶️ PASO 8 — Cargar CUIs e inicializar el navegador

In [ ]:
print("📂 Cargando CUIs...")
df_cui = pd.read_excel(RUTA_ENTRADA)
lista_completa = df_cui['CUI'].astype(str).tolist()
print(f"📊 Total CUIs en archivo: {len(lista_completa)}")

resultados_previos = cargar_progreso()
lista_cuis, procesados_dict = obtener_pendientes(lista_completa, resultados_previos)

if not lista_cuis:
    print("🎉 ¡Todos los CUIs ya fueron procesados exitosamente!")
    print("💡 Para reprocesar CUIs fallidos, elimina el archivo de salida y vuelve a ejecutar.")
else:
    resultados = resultados_previos.copy()
    driver = crear_driver()
    print(f"\n⚡ Listo para procesar {len(lista_cuis)} CUIs pendientes")

### ▶️ PASO 9 — Ejecutar el scraping
> ⚠️ **No cierres Chrome** mientras se ejecuta.
> Puedes interrumpir con el botón **⏹️ Stop** de Jupyter; el progreso se guardará automáticamente.
> Al volver a ejecutar los PASOS 8 y 9, continuará desde donde se quedó.

In [ ]:
tiempo_inicio = time.time()
exitosos = 0
fallos   = 0

print(f"{'='*60}")
print(f"Inicio: {time.strftime('%H:%M:%S')}")
print(f"{'='*60}\n")

try:
    for idx, cui in enumerate(lista_cuis, 1):
        t_inicio  = time.time()
        es_retry  = str(cui) in procesados_dict
        marca     = "🔄" if es_retry else "🆕"
        print(f"{marca} [{idx}/{len(lista_cuis)}] {cui}:", end=" ")

        try:
            datos = procesar_cui(driver, cui)
            actualizar_resultado(resultados, cui, datos)
            exitosos += 1
            print(f"✅ ({time.time()-t_inicio:.1f}s)")
        except Exception:
            agregar_fallo(resultados, cui)
            fallos += 1
            print(f"❌ ({time.time()-t_inicio:.1f}s)")

        # Guardar progreso periódicamente
        if (idx % GUARDAR_CADA == 0) or (idx == len(lista_cuis)):
            guardar(resultados)
            if idx % GUARDAR_CADA == 0:
                print(f"   💾", end="")

        # Mostrar estadísticas cada 10 CUIs
        if idx % 10 == 0:
            t_el = time.time() - tiempo_inicio
            eta  = (len(lista_cuis) - idx) * (t_el / idx)
            print(f"\n   📊 {exitosos}✅ {fallos}❌ | {t_el:.0f}s transcurridos | ETA: {eta:.0f}s\n")

except KeyboardInterrupt:
    print("\n\n⚠️ INTERRUMPIDO — guardando progreso...")
    guardar(resultados)
    driver.quit()
    print("💾 Progreso guardado. Re-ejecuta los PASOS 8 y 9 para continuar.")

else:
    driver.quit()
    guardar(resultados)
    t_total = time.time() - tiempo_inicio
    print(f"\n{'='*60}")
    print("🏁 COMPLETADO")
    print(f"{'='*60}")
    print(f"📊 CUIs procesados en esta sesión: {len(lista_cuis)}")
    print(f"✅ Exitosos : {exitosos}")
    print(f"❌ Fallidos : {fallos}")
    print(f"⏱️  Tiempo  : {t_total:.1f}s ({t_total/60:.1f} min)")
    print(f"💾 Guardado en: {RUTA_SALIDA}")

### 📊 PASO 10 — Ver estadísticas del archivo final (opcional)

In [ ]:
if RUTA_SALIDA.exists():
    df_final  = pd.read_excel(RUTA_SALIDA)
    col_costo = 'Costo Inversión Total (a)'
    total          = len(df_final)
    con_datos      = len(df_final[df_final[col_costo] != 'NO DISPONIBLE'])
    sin_datos      = total - con_datos
    print(f"{'='*60}")
    print(f"📊 ESTADÍSTICAS TOTALES DEL ARCHIVO")
    print(f"{'='*60}")
    print(f"   Total CUIs : {total}")
    print(f"   ✅ Con datos: {con_datos} ({con_datos/total*100:.1f}%)")
    print(f"   ❌ Sin datos: {sin_datos} ({sin_datos/total*100:.1f}%)")
    print(f"   📁 Archivo  : {RUTA_SALIDA}")
    print(f"{'='*60}")
    df_final.head(10)
else:
    print("⚠️ No se encontró el archivo de resultados. Ejecuta el scraping primero.")

---
---
## 📦 PARTE 2 — Avance Físico (%)
### Extrae el % de Avance Físico desde la URL directa de Invierte.pe
> **Nota:** Puedes ejecutar esta parte de forma independiente de la Parte 1.

### ⚙️ PASO 1 — Configuración del Avance Físico
> **⚠️ Esta es la ÚNICA celda que debes editar para esta parte.**

In [ ]:
# Si ejecutas esta parte SIN haber ejecutado la Parte 1,
# descomenta las dos líneas siguientes:
# from pathlib import Path
# CARPETA_BASE = Path.cwd()

# ================================================================
# CONFIGURACIÓN PARTE 2 — Solo edita esta sección
# ================================================================

NOMBRE_ENTRADA_AF = "CUI.xlsx"                       # <- Archivo con la lista de CUIs
NOMBRE_SALIDA_AF  = "resultados_avance_fisico.xlsx"  # <- Archivo de resultados

RUTA_ENTRADA_AF = CARPETA_BASE / NOMBRE_ENTRADA_AF
RUTA_SALIDA_AF  = CARPETA_BASE / NOMBRE_SALIDA_AF

MAX_REINTENTOS_AF  = 2
MODO_VISIBLE_AF    = True   # True = Chrome visible | False = silencioso
GUARDAR_CADA_AF    = 5
TIMEOUT_PAGINA_AF  = 20
TIMEOUT_ELEM_AF    = 10

print(f"📥 Archivo de entrada: {NOMBRE_ENTRADA_AF}")
print(f"📤 Archivo de salida : {NOMBRE_SALIDA_AF}")
if RUTA_ENTRADA_AF.exists():
    print(f"✅ '{NOMBRE_ENTRADA_AF}' encontrado")
else:
    print(f"❌ ERROR: No se encontró '{NOMBRE_ENTRADA_AF}' en {CARPETA_BASE}")

### ⚙️ PASO 2 — Importar librerías (si no ejecutaste la Parte 1)

In [ ]:
# Si ya ejecutaste la Parte 1, las librerías ya están importadas.
# Si ejecutas esta parte de forma independiente, descomenta el bloque:

# import pandas as pd
# import time
# from selenium import webdriver
# from selenium.webdriver import Chrome
# from selenium.webdriver.chrome.service import Service
# from selenium.webdriver.common.by import By
# from selenium.webdriver.support.ui import WebDriverWait as Wait
# from selenium.webdriver.support import expected_conditions as EC
# from selenium.common.exceptions import TimeoutException
# from webdriver_manager.chrome import ChromeDriverManager

print("✅ Librerías listas")

### ⚙️ PASO 3 — Funciones de checkpoint y scraping del Avance Físico

In [ ]:
def cargar_progreso_af():
    """Carga el progreso previo del Avance Físico."""
    if RUTA_SALIDA_AF.exists():
        try:
            df = pd.read_excel(RUTA_SALIDA_AF)
            print(f"📥 Progreso encontrado: {len(df)} CUIs ya procesados")
            return df.to_dict('records')
        except Exception as e:
            print(f"⚠️ Error al leer progreso: {e}")
    return []

def obtener_pendientes_af(completa, procesados):
    """Calcula los CUIs pendientes para Avance Físico."""
    if not procesados:
        print(f"📋 Todos los CUIs pendientes: {len(completa)}")
        return completa
    cuis_ok = {str(r['CUI']) for r in procesados}
    pendientes = [cui for cui in completa if str(cui) not in cuis_ok]
    if pendientes:
        print(f"⏳ Pendientes: {len(pendientes)}")
    return pendientes

def guardar_af(resultados):
    """Guarda el progreso del Avance Físico."""
    try:
        pd.DataFrame(resultados).to_excel(RUTA_SALIDA_AF, index=False)
    except Exception as e:
        print(f"⚠️ Error al guardar: {e}")

def extraer_avance_fisico(driver):
    """Extrae el % de Avance Físico del elemento 'por_afis'."""
    try:
        Wait(driver, TIMEOUT_ELEM_AF).until(
            EC.presence_of_element_located((By.ID, "por_afis"))
        )
        avance = driver.execute_script("""
            var elem = document.getElementById('por_afis');
            if (elem) return (elem.textContent || elem.innerText || '').trim();
            return 'NO DISPONIBLE';
        """)
        return avance if (avance and avance.strip() and avance != 'NO DISPONIBLE') else 'NO DISPONIBLE'
    except TimeoutException:
        print("⏱️ Timeout", end=" ")
        return 'NO DISPONIBLE'
    except Exception as e:
        print(f"⚠️ Error: {str(e)[:30]}", end=" ")
        return 'NO DISPONIBLE'

def procesar_cui_af(driver, cui, intento=1):
    """Navega a la URL directa del CUI y extrae el % Avance Físico."""
    try:
        url = f"https://ofi5.mef.gob.pe/invierteWS/Repseguim/RepEstimac?codigo={cui}"
        driver.get(url)
        time.sleep(0.5)
        avance = extraer_avance_fisico(driver)
        if avance == 'NO DISPONIBLE':
            raise Exception("No se pudo extraer el avance físico")
        return avance
    except Exception as e:
        if intento < MAX_REINTENTOS_AF:
            print(f"🔄{intento+1}...", end=" ")
            time.sleep(1)
            return procesar_cui_af(driver, cui, intento + 1)
        else:
            raise e

print("✅ Funciones de Avance Físico listas")

### ▶️ PASO 4 — Cargar CUIs e inicializar el navegador

In [ ]:
print("📂 Cargando CUIs...")
df_cui_af = pd.read_excel(RUTA_ENTRADA_AF)
lista_completa_af = df_cui_af['CUI'].astype(str).tolist()
print(f"📊 Total CUIs en archivo: {len(lista_completa_af)}")

resultados_af = cargar_progreso_af()
lista_cuis_af = obtener_pendientes_af(lista_completa_af, resultados_af)

if not lista_cuis_af:
    print("🎉 ¡Todos los CUIs ya fueron procesados!")
else:
    # Si ya ejecutaste la Parte 1 y crear_driver() está definida, se reutiliza.
    # Si no, se crea un driver temporal con los parámetros de esta parte.
    _opts = webdriver.ChromeOptions()
    if not MODO_VISIBLE_AF:
        _opts.add_argument("--headless=new")
    else:
        _opts.add_argument("--start-maximized")
    _opts.add_argument("--disable-gpu")
    _opts.add_argument("--no-sandbox")
    _opts.add_argument("--disable-dev-shm-usage")
    _opts.add_argument("--disable-extensions")
    _opts.add_argument("--disable-logging")
    _opts.add_argument("--log-level=3")
    _opts.add_experimental_option('excludeSwitches', ['enable-logging'])
    _opts.page_load_strategy = 'normal'
    driver_af = Chrome(service=Service(ChromeDriverManager().install()), options=_opts)
    driver_af.set_page_load_timeout(TIMEOUT_PAGINA_AF)
    driver_af.set_script_timeout(15)
    print(f"\n⚡ Listo para procesar {len(lista_cuis_af)} CUIs...")

### ▶️ PASO 5 — Ejecutar scraping del Avance Físico
> ⚠️ No cierres Chrome. Si se interrumpe, re-ejecuta los PASOS 4 y 5 para continuar.

In [ ]:
tiempo_inicio_af = time.time()
exitosos_af = 0
fallos_af   = 0

print(f"{'='*60}")
print(f"Inicio: {time.strftime('%H:%M:%S')}")
print(f"{'='*60}\n")

try:
    for idx, cui in enumerate(lista_cuis_af, 1):
        t_inicio = time.time()
        print(f"[{idx}/{len(lista_cuis_af)}] {cui}:", end=" ")

        try:
            avance = procesar_cui_af(driver_af, cui)
            resultados_af.append({"CUI": cui, "% Avance Físico": avance})
            exitosos_af += 1
            print(f"✅ {avance} ({time.time()-t_inicio:.1f}s)")
        except Exception:
            resultados_af.append({"CUI": cui, "% Avance Físico": "NO DISPONIBLE"})
            fallos_af += 1
            print(f"❌ ({time.time()-t_inicio:.1f}s)")

        if (idx % GUARDAR_CADA_AF == 0) or (idx == len(lista_cuis_af)):
            guardar_af(resultados_af)
            if idx % GUARDAR_CADA_AF == 0:
                print(f"   💾", end="")

        if idx % 10 == 0:
            t_el = time.time() - tiempo_inicio_af
            eta  = (len(lista_cuis_af) - idx) * (t_el / idx)
            print(f"\n   📊 {exitosos_af}✅ {fallos_af}❌ | {t_el:.0f}s | ETA: {eta:.0f}s\n")

except KeyboardInterrupt:
    print("\n\n⚠️ INTERRUMPIDO — guardando progreso...")
    guardar_af(resultados_af)
    driver_af.quit()
    print("💾 Progreso guardado. Re-ejecuta los PASOS 4 y 5 para continuar.")

else:
    driver_af.quit()
    guardar_af(resultados_af)
    t_total_af = time.time() - tiempo_inicio_af
    print(f"\n{'='*60}")
    print("🏁 COMPLETADO — AVANCE FÍSICO")
    print(f"{'='*60}")
    print(f"📊 Total procesados: {len(lista_cuis_af)}")
    print(f"✅ Exitosos : {exitosos_af}")
    print(f"❌ Fallidos : {fallos_af}")
    print(f"⏱️  Tiempo  : {t_total_af:.1f}s ({t_total_af/60:.1f} min)")
    print(f"💾 Guardado en: {RUTA_SALIDA_AF}")